<a href="https://www.kaggle.com/code/nhkhoi/notebook32d3614afa?scriptVersionId=300881669" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/llm-agentic-legal-information-retrieval/sample_submission.csv
/kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv
/kaggle/input/competitions/llm-agentic-legal-information-retrieval/val.csv
/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv
/kaggle/input/competitions/llm-agentic-legal-information-retrieval/train.csv
/kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv


In [2]:
!pip install sentence_transformers bm25s

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.3 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import time
import warnings
import gc
import sys
import time
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer, CrossEncoder
import bm25s


In [4]:
warnings.filterwarnings('ignore')


train_path = '/kaggle/input/llm-agentic-legal-information-retrieval/train.csv'
val_path = '/kaggle/input/llm-agentic-legal-information-retrieval/val.csv'
test_path = '/kaggle/input/llm-agentic-legal-information-retrieval/test.csv'
laws_path = '/kaggle/input/llm-agentic-legal-information-retrieval/laws_de.csv'
court_path = '/kaggle/input/llm-agentic-legal-information-retrieval/court_considerations.csv'
output_path = 'submission.csv'

In [5]:
embedding_model_name = 'BAAI/bge-m3'
reranker_model_name = 'BAAI/bge-reranker-v2-m3'

use_gpu = torch.cuda.is_available()
num_gpus = torch.cuda.device_count() if use_gpu else 0

if num_gpus >= 2:
    device_emb = 'cuda:0'
    device_rerank = 'cuda:1'
elif num_gpus == 1:
    device_emb = 'cuda:0'
    device_rerank = 'cuda:0'
else:
    device_emb = 'cpu'
    device_rerank = 'cpu'

In [6]:
laws_batch_size = 8
query_batch_size = 512
max_length = 512
top_k_retrieval = 60
top_k_final = 10
text_truncate = 2000

use_hybrid = True
use_reranking = True
use_query_expansion = True

In [7]:
def load_data():
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)
    laws_df = pd.read_csv(laws_path)
    laws_df['text'] = laws_df['text'].fillna('')
    court_df = pd.read_csv(court_path, usecols=['citation', 'text'])
    court_df['text'] = court_df['text'].fillna('')
    
    print(f"loaded {len(train_df)} train, {len(val_df)} val, {len(test_df)} test queries")
    print(f"loaded {len(laws_df)} laws, {len(court_df)} court decisions")
    
    return train_df, val_df, test_df, laws_df, court_df

In [8]:
def tokenize_for_bm25(texts):
    truncated = [str(t).lower()[:text_truncate] for t in texts]
    return bm25s.tokenize(truncated, stopwords='de')

def build_bm25_index(texts):
    tokens = tokenize_for_bm25(texts)
    retriever = bm25s.BM25()
    retriever.index(tokens)
    del tokens
    gc.collect()
    print("index ready")
    return retriever

def search_bm25(retriever, query, top_k):
    query_tokens = bm25s.tokenize([query.lower()], stopwords='de')
    docs, scores = retriever.retrive(query_tokens, k=top_k)
    results = [(docs[0][i], scores[0][i]) for i in range(len(docs[0]))]
    return results

In [9]:
def load_embedding_model():
    print(f"loading embedding model on {device_emb}")
    model = SentenceTransformer(
        embedding_model_name,
        device=device_emb,
        model_kwargs={'torch_dtype': torch.float16}
    )
    print("model ready")
    return model

In [10]:
def encode_corpus(model, texts):
    truncated = [str(t)[:text_truncate] for t in texts]
    embeddings = model.encode(
        truncated,
        batch_size=laws_batch_size,
        convert_to_tensor=False,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return embeddings

def encode_query(model, query):
    embedding = model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )
    return embedding.cpu().numpy()

def search_dense(query_embedding, corpus_embeddings, top_k):
    scores = np.dot(corpus_embeddings, query_embedding)
    top_indices = np.argsort(scores)[-top_k:][::-1]
    results = [(int(idx), float(scores[idx])) for idx in top_indices]
    return results

def load_reranker():
    print(f"loading reranker on {device_rerank}")
    model = CrossEncoder(
        reranker_model_name,
        device=device_rerank,
        max_length=512,
        automodel_args={'torch_dtype': torch.float16}
    )
    print("model ready")
    return model

def rerank_candidates(reranker, query, candidates):
    if not candidates:
        return []
    
    pairs = []
    citations = []
    
    for citation, text, _ in candidates:
        truncated = str(text)[:text_truncate]
        pairs.append([query, truncated])
        citations.append(citation)
    
    scores = reranker.predict(pairs, batch_size=8, show_progress_bar=False)
    scored = list(zip(citations, scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    
    return [c[0] for c in scored]